# E. coli reconstruction from reduced reads
Run **Runtime → Run all** on a standard CPU runtime. No GPU or paid plan is required by this notebook. Resources and execution time depend on Colab availability.

This is a **Flye baseline benchmark, not model training**. It uses public E. coli reads and compares three read amounts with a manufacturer reference. No lab MIC data or antibiotic predictions are included. These are research inputs for predictions of in-vitro susceptibility, not prescribing advice.

The dataset is approximately 308 MB compressed, plus a 50 MB reference archive. Temporary files live in `/content/amr_ecoli`. Download results before the runtime expires. The notebook never mounts your Google Drive or reads your private repository.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, zipfile, runpy
ROOT = Path('/content/amr_ecoli')
ROOT.mkdir(parents=True, exist_ok=True)
THREADS = min(4, os.cpu_count() or 1)
def run(args, **kwargs):
    print('Running:', ' '.join(map(str,args)), flush=True)
    with subprocess.Popen(list(map(str,args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, args)
ENV = ROOT / 'env'
run([sys.executable, '-m', 'pip', 'install', 'virtualenv'])
run([sys.executable, '-m', 'virtualenv', ENV])
PYTHON = ENV / 'bin/python'
run([PYTHON, '-m', 'pip', 'install', 'snakemake==9.27.0'])
FLYE = ROOT / 'Flye'
if not FLYE.exists():
    run(['git', 'clone', '--depth', '1', '--branch', '2.9.6', 'https://github.com/mikolmogorov/Flye.git', FLYE])
commit = subprocess.check_output(['git', '-C', str(FLYE), 'rev-parse', 'HEAD'], text=True).strip()
assert commit == '886b8c17412cdf3a2868a28237bca6c5ad1da156', commit
run(['make', '-C', FLYE])
run([PYTHON, FLYE / 'bin/flye', '--version'])


In [ ]:
def download(url, target, expected_hash):
    if not target.exists():
        partial = target.with_suffix(target.suffix + '.part')
        run(['curl', '-fL', '--retry', '3', url, '-o', partial])
        partial.rename(target)
    with target.open('rb') as handle:
        actual = hashlib.file_digest(handle, 'sha256').hexdigest()
    if actual != expected_hash:
        raise ValueError(f'Checksum mismatch: {target}. Do not use this download.')
    print(target.name, 'verified')

download('https://ont-exd-int-s3-euwst1-epi2me-labs.s3-eu-west-1.amazonaws.com/assembly_tutorial/ecoli.60x.r103.fastq.gz', ROOT / 'reads.fastq.gz', 'b13d9321b2dc1f7073dd0a7f947f7bfc98f9d04b1344c52ee08cc92f1c6f4a64')
download('https://zymo-files.s3.amazonaws.com/BioPool/ZymoBIOMICS.STD.refseq.v3.zip', ROOT / 'zymo-reference.zip', '9c7002d4f353d8b6f55ca41e6f162c59e5524f145136a4e9b7eae3409555749d')
with zipfile.ZipFile(ROOT / 'zymo-reference.zip') as archive:
    (ROOT / 'reference.fasta').write_bytes(archive.read('ZymoBIOMICS.STD.refseq.v3/Genomes/Escherichia_coli_complete_genome.fna'))


In [ ]:
files = {'Snakefile': "# Run from this experiment directory; tool paths are supplied explicitly.\nFLYE = config['flye']\nMINIMAP = config['minimap']\nPYTHON = config.get('python', 'python3')\nLEVELS = ['00', '01', '02']\nrule all:\n    input: expand('results/{level}/metrics.json', level=LEVELS)\nrule assemble:\n    input: 'subsets/subset_{level}.fastq'\n    output: 'results/{level}/assembly.fasta'\n    log: 'results/{level}/run.log'\n    threads: 4\n    shell:\n        '{PYTHON:q} {FLYE:q} --nano-raw {input:q} --threads {threads} --out-dir results/{wildcards.level} > {log:q} 2>&1'\nrule align:\n    input: asm='results/{level}/assembly.fasta', ref='reference.fasta'\n    output: 'results/{level}/alignment.paf'\n    log: 'results/{level}/alignment.log'\n    threads: 4\n    shell:\n        '{MINIMAP:q} -x asm5 -c --secondary=no -t {threads} {input.ref:q} {input.asm:q} > {output:q} 2> {log:q}'\nrule metrics:\n    input: asm='results/{level}/assembly.fasta', ref='reference.fasta', paf='results/{level}/alignment.paf'\n    output: 'results/{level}/metrics.json'\n    shell:\n        '{PYTHON:q} summarize.py {input.ref:q} {input.asm:q} {input.paf:q} > {output:q}'\n", 'summarize.py': '"""Descriptive reference alignment metrics, not AMR or clinical validation."""\nimport json,sys\nfrom pathlib import Path\n\ndef fasta(path):\n    out={}; name=None\n    for line in Path(path).read_text().splitlines():\n        if line.startswith(\'>\'):\n            name=line[1:].split()[0]; out[name]=\'\'\n        elif line.strip():\n            if name is None: raise ValueError(\'Invalid FASTA\')\n            out[name]+=line.strip()\n    return out\n\ndef union_length(intervals):\n    total=0; end=0\n    for a,b in sorted(intervals):\n        total+=max(0,b-max(a,end)); end=max(end,b)\n    return total\n\ndef summarize(ref_path,asm_path,paf_path):\n    refs=fasta(ref_path); asm=fasta(asm_path)\n    targets={k:[] for k in refs}; queries={k:[] for k in asm}\n    matches=columns=0\n    for line in Path(paf_path).read_text().splitlines():\n        p=line.split(\'\\t\')\n        if \'tp:A:P\' not in p[12:]: continue\n        targets[p[5]].append((int(p[7]),int(p[8])))\n        queries[p[0]].append((int(p[2]),int(p[3])))\n        matches+=int(p[9]); columns+=int(p[10])\n    ref_total=sum(map(len,refs.values())); asm_total=sum(map(len,asm.values()))\n    return dict(contigs=len(asm),assembly_bases=asm_total,reference_bases=ref_total,\n        reference_covered_percent=100*sum(map(union_length,targets.values()))/ref_total,\n        assembly_aligned_percent=100*sum(map(union_length,queries.values()))/asm_total if asm_total else 0,\n        aligned_identity_percent=100*matches/columns if columns else None,\n        per_reference_covered_percent={k:100*union_length(targets[k])/len(v) for k,v in refs.items()},\n        note=\'Primary alignment-span coverage and alignment-weighted identity; not a structural-error or AMR assessment.\')\n\nif __name__==\'__main__\':\n    print(json.dumps(summarize(*sys.argv[1:]),indent=2))\n', 'test_summary.py': "import tempfile\nimport unittest\nfrom pathlib import Path\nfrom summarize import summarize, union_length\n\nclass SummaryTests(unittest.TestCase):\n    def test_overlapping_intervals_not_double_counted(self):\n        self.assertEqual(union_length([(0,10),(5,15),(20,25)]),20)\n        self.assertEqual(union_length([]),0)\n\n    def test_known_alignment_and_missing_replicon(self):\n        with tempfile.TemporaryDirectory() as d:\n            p=Path(d)\n            (p/'r').write_text('>r\\n'+'A'*100+'\\n>plasmid\\n'+'A'*100+'\\n')\n            (p/'a').write_text('>a\\n'+'A'*80+'\\n')\n            (p/'p').write_text('a\\t80\\t0\\t80\\t+\\tr\\t100\\t10\\t90\\t79\\t80\\t60\\ttp:A:P\\n')\n            m=summarize(p/'r',p/'a',p/'p')\n            self.assertEqual(m['reference_covered_percent'],40)\n            self.assertEqual(m['aligned_identity_percent'],98.75)\n            self.assertEqual(m['per_reference_covered_percent']['plasmid'],0)\n            (p/'p').write_text('')\n            m=summarize(p/'r',p/'a',p/'p')\n            self.assertEqual(m['reference_covered_percent'],0)\n            self.assertIsNone(m['aligned_identity_percent'])\n\nif __name__=='__main__': unittest.main()\n", 'collect_results.py': '"""Collect completed pilot metrics without silently excluding failed runs."""\nimport json\nfrom pathlib import Path\nfrom summarize import fasta\n\nroot=Path(__file__).resolve().parent\nmanifest=json.loads((root/\'subsets/manifest.json\').read_text())\nref_length=sum(map(len,fasta(root/\'reference.fasta\').values()))\nrows=[]\nfor i,subset in enumerate(manifest[\'subsets\']):\n    path=root/f\'results/{i:02d}/metrics.json\'\n    rows.append(dict(fraction=subset[\'fraction\'],reads=subset[\'reads\'],bases=subset[\'bases\'],\n        average_depth=subset[\'bases\']/ref_length,status=\'completed\' if path.exists() else \'missing_or_failed\',\n        metrics=json.loads(path.read_text()) if path.exists() else None))\n(root/\'benchmark_results.json\').write_text(json.dumps(rows,indent=2)+\'\\n\')\nlines=[\'# Pilot results\',\'\', \'Flye 2.9.6; one E. coli mock-community sample, one subsampling seed. No model training or MIC evaluation.\', \'\', \'| Reads used | Average depth | Contigs | Reference aligned breadth | Aligned sequence identity |\', \'|---|---:|---:|---:|---:|\']\nfor r in rows:\n    m=r[\'metrics\']\n    if m is None:\n        lines.append(f"| {r[\'fraction\']:.0%} | {r[\'average_depth\']:.2f}x | missing/failed | — | — |")\n    else:\n        identity=\'unavailable\' if m[\'aligned_identity_percent\'] is None else f"{m[\'aligned_identity_percent\']:.4f}%"\n        lines.append(f"| {r[\'fraction\']:.0%} | {r[\'average_depth\']:.2f}x | {m[\'contigs\']} | {m[\'reference_covered_percent\']:.4f}% | {identity} |")\nlines+=[\'\',\'Alignment breadth and identity are different measures. These results do not establish complete correctness, resistance-marker preservation, MIC accuracy, or a validated minimum sequencing depth. See README.md for provenance and limitations.\',\'\']\n(root/\'RESULTS.md\').write_text(\'\\n\'.join(lines))\nprint(\'\\n\'.join(lines))\n', 'README.md': "# E. coli reduced-read reconstruction pilot\n\nThis is a Flye assembly benchmark, not neural-model training and not a validated\nMIC predictor. No pretrained DNA language model is assumed to recover unobserved\nresistance genes. The output is an assembly based on observed read overlaps.\nThese are research inputs for future predictions of in-vitro susceptibility,\nnot prescribing advice.\n\n## Baseline and data\n\n- Flye 2.9.6, commit 886b8c17412cdf3a2868a28237bca6c5ad1da156.\n- Use `--nano-raw`, following this older R10.3 tutorial's command; do not assume\n  these are current R10 SUP reads suitable for the latest high-quality presets.\n- Four CPU threads. No GPU or learned weights required.\n- Reads: https://ont-exd-int-s3-euwst1-epi2me-labs.s3-eu-west-1.amazonaws.com/assembly_tutorial/ecoli.60x.r103.fastq.gz\n- Tutorial: https://epi2me.nanoporetech.com/notebooks/Assembly_Tutorial.html\n- Reference: E. coli FASTA extracted from the manufacturer's bundle at\n  https://zymo-files.s3.amazonaws.com/BioPool/ZymoBIOMICS.STD.refseq.v3.zip\n- The tutorial's historical reference URL returns 404. The manufacturer's v3\n  reference is a replacement for the same mock-community organism, not a verified\n  byte-identical copy of the historical reference. Reference discrepancies can\n  contribute to measured differences.\n\nThe reference contains a 4,804,267 bp chromosome and a 120,874 bp plasmid. The\nreference is used only for evaluation, not to guide assembly. Tutorial reads were\nalready selected by reference alignment from a mock community: this introduces\nselection bias and does not evaluate mixed blood-culture samples, contamination,\nmodern chemistry, or a general collection of clinical isolates.\n\n## Experiment\n\nSeed 42, random nested read fractions 0.25, 0.5 and 1.0. These nominally represent\n15x, 30x and 60x based on the source description. Actual read bases divided by\nreference length are reported separately and are not uniform per-position coverage.\nThis is one isolate and one subsampling seed, not model selection or external\nvalidation. There is no training/test split because no parameters are learned here.\nDo not use these results as independent biological replicates.\n\n## Reproduce\n\nInstall/build Flye as described at https://github.com/mikolmogorov/Flye/blob/flye/docs/INSTALL.md\nand install Snakemake in an isolated environment. The pilot used Snakemake 9.27.0.\nCompile Flye with its top-level `make` (not parallel top-level make: its bundled\nsamtools configuration can race against minimap2 compilation).\n\nPlace `reads.fastq.gz` and `reference.fasta` in this directory. From the repo root:\n\n```bash\nPYTHONPATH=src python3 -m genome2mic.partial.subsample experiments/ecoli_reconstruction/reads.fastq.gz experiments/ecoli_reconstruction/subsets --fractions 0.25 0.5 1 --seed 42\n```\n\nThen, from this experiment directory:\n\n```bash\nsnakemake --cores 4 --config flye=/absolute/path/Flye/bin/flye minimap=/absolute/path/Flye/bin/flye-minimap2 python=python3\npython3 -m unittest test_summary -v\n```\n\nLarge raw data, tool builds, intermediate assemblies, and Snakemake state are\nignored by Git. The metrics and provenance snapshot can be committed separately.\n\n## Interpreting metrics\n\n- Reference covered: union of primary alignment spans divided by reference length.\n  This measures aligned breadth, not the fraction of all bases proven correct.\n- Aligned identity: matches / alignment block length from minimap2 `-c` primary PAF\n  records. This is alignment-weighted and not an independent complete-genome score.\n- Per-reference breadth: report the chromosome and plasmid separately.\n- Contigs: number of assembled sequence pieces; fewer alone does not prove accuracy.\n\nNo structural-error benchmark, AMR gene/variant audit, learned calibration, MIC\nprediction, or clinical evaluation is included. Later comparisons need multiple\nisolates/seeds, AMR feature preservation, and ultimately lab MIC labels. Gaps must\nremain unknown; reference sequence must not be copied into missing regions.\n", 'subsample.py': '"""Create nested subsets of single-end four-line FASTQ reads; no MIC prediction."""\nimport argparse\nimport gzip\nimport json\nimport math\nimport random\nfrom contextlib import ExitStack\nfrom pathlib import Path\n\n\ndef records(path):\n    opener = gzip.open if path.suffix == ".gz" else open\n    with opener(path, "rt") as handle:\n        while True:\n            header = handle.readline()\n            if not header:\n                return\n            sequence, plus, quality = (handle.readline() for _ in range(3))\n            seq, qual = sequence.rstrip("\\r\\n"), quality.rstrip("\\r\\n")\n            if (not header.startswith("@") or not plus.startswith("+")\n                    or not seq or not quality or len(seq) != len(qual)):\n                raise ValueError("Malformed FASTQ: expected four-line records with equal sequence/quality lengths")\n            yield header + sequence + plus + quality, len(seq)\n\n\ndef subsample(source, output, fractions, seed=42, mode="random"):\n    source, output = Path(source), Path(output)\n    fractions = sorted(set(fractions))\n    if not fractions or any(not math.isfinite(f) or not 0 < f <= 1 for f in fractions):\n        raise ValueError("Fractions must be finite numbers in (0, 1]")\n    if mode not in {"random", "prefix"}:\n        raise ValueError("Mode must be random or prefix")\n    # Validate the entire input before creating outputs; count for prefix mode.\n    total_reads = total_bases = 0\n    for _, bases in records(source):\n        total_reads += 1\n        total_bases += bases\n    if not total_reads:\n        raise ValueError("Input FASTQ is empty")\n    output.mkdir(parents=True, exist_ok=False)\n    rng = random.Random(seed)\n    entries = [dict(fraction=f, file=f"subset_{i:02d}.fastq", reads=0, bases=0)\n               for i, f in enumerate(fractions)]\n    with ExitStack() as stack:\n        handles = [stack.enter_context((output / e["file"]).open("w")) for e in entries]\n        for index, (record, bases) in enumerate(records(source)):\n            draw = rng.random()\n            for entry, handle in zip(entries, handles):\n                keep = (draw < entry["fraction"] if mode == "random"\n                        else index < math.ceil(entry["fraction"] * total_reads))\n                if keep:\n                    handle.write(record)\n                    entry["reads"] += 1\n                    entry["bases"] += bases\n    manifest = dict(source=str(source.resolve()), mode=mode, seed=seed,\n                    total_reads=total_reads, total_bases=total_bases, subsets=entries)\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("source", type=Path)\n    parser.add_argument("output", type=Path, help="New directory; existing directories are refused")\n    parser.add_argument("--fractions", type=float, nargs="+", default=[.01, .05, .1, .25, .5, 1.0])\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--mode", choices=["random", "prefix"], default="random")\n    args = parser.parse_args()\n    manifest = subsample(args.source, args.output, args.fractions, args.seed, args.mode)\n    print(json.dumps(manifest, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, contents in files.items():
    (ROOT / name).write_text(contents)
run([PYTHON, '-m', 'unittest', 'test_summary', '-v'], cwd=ROOT)
if not (ROOT / 'subsets').exists():
    run([PYTHON, ROOT/'subsample.py', ROOT/'reads.fastq.gz', ROOT/'subsets', '--fractions', '0.25', '0.5', '1', '--seed', '42'])
else:
    print('Reusing existing subsets; do not change fraction settings without a new output directory.')
print((ROOT / 'subsets/manifest.json').read_text())


## Run the assemblies
This can take several minutes or longer. It uses the CPU. Each assembly is followed by reference alignment and descriptive metrics. Re-running resumes completed workflow outputs. Assembly errors should be inspected, not hidden.


In [ ]:
run([ENV/'bin/snakemake', '--cores', THREADS, '--rerun-incomplete', '--config',
     f'flye={FLYE}/bin/flye', f'minimap={FLYE}/bin/flye-minimap2', f'python={PYTHON}'], cwd=ROOT)
run([PYTHON, ROOT/'collect_results.py'], cwd=ROOT)
from IPython.display import display, Markdown
display(Markdown((ROOT/'RESULTS.md').read_text()))


In [ ]:
provenance = {
    'tool':'Flye', 'commit':commit, 'threads':THREADS, 'seed':42,
    'fractions':[0.25,0.5,1], 'platform':'Colab Linux', 'trained_model':False,
    'read_sha256':hashlib.file_digest((ROOT/'reads.fastq.gz').open('rb'),'sha256').hexdigest(),
    'reference_sha256':hashlib.file_digest((ROOT/'reference.fasta').open('rb'),'sha256').hexdigest()
}
(ROOT/'provenance.json').write_text(json.dumps(provenance,indent=2))
archive_path = ROOT/'ecoli_reconstruction_results.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for name in ['RESULTS.md','benchmark_results.json','provenance.json','README.md','Snakefile','summarize.py','subsets/manifest.json']:
        archive.write(ROOT/name,name)
    for path in (ROOT/'results').glob('*/*'):
        if path.is_file() and path.name in {'assembly.fasta','assembly_info.txt','metrics.json','run.log','alignment.paf'}:
            archive.write(path,path.relative_to(ROOT))
from google.colab import files as colab_files
colab_files.download(str(archive_path))


## Interpretation
Reference-aligned breadth is not base accuracy. High aligned identity does not prove AMR markers or mutations are correct. This is one mock-community isolate, one subset seed, and older R10.3 data preselected by alignment. It does not validate early blood-culture sequencing or downstream MIC prediction. See the embedded README for data provenance and limitations.
